# 第 1 讲：Python Notebook 与 AI 辅助数据分析

本练习由 Codex 辅助准备。所有数据均为人工模拟的教学数据，不来自真实患者，不用于医学判断。

运行方法：右上角选择 `.venv` 中的 Python，然后点击 **Run All**。阅读每个输出，并确认你能解释每一步。


In [1]:
import sys
print("Python 版本：", sys.version.split()[0])
print("当前 Python：", sys.executable)
print("我的第一个 Notebook 已成功运行！")

Python 版本： 3.12.15
当前 Python： D:\博士38720260158062\AI使用\AI4BiomedicalDS\.venv\Scripts\python.exe
我的第一个 Notebook 已成功运行！


## 数据列的含义

| 变量 | 含义 | 单位 / 类型 |
|---|---|---|
| sample_id | 模拟样本编号 | 标识符，不计算均值 |
| age_years | 模拟年龄 | 年，数值变量；有一个缺失值 |
| group | 模拟分组 | A / B，分类变量 |
| systolic_bp_mmHg | 模拟收缩压 | mmHg，数值变量 |
| glucose_mmol_L | 模拟血糖值 | mmol/L，数值变量 |

真实分析时，应核对数据提供者的变量说明；AI 不能只凭列名确定实际含义。


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

data_path = Path("data/demo.csv")
df = pd.read_csv(data_path, dtype={"sample_id": "string", "group": "string"})
display(df)
print(f"样本数：{df.shape[0]}；列数：{df.shape[1]}")

,sample_id,age_years,group,systolic_bp_mmHg,glucose_mmol_L
0,S01,24.0,A,112,4.7
1,S02,31.0,B,128,5.2
2,S03,45.0,A,135,5.8
3,S04,52.0,B,142,6.1
4,S05,NaN,A,118,4.9
5,S06,38.0,B,125,5.4
6,S07,29.0,A,110,4.5
7,S08,61.0,B,150,6.3


样本数：8；列数：5


## 描述性统计与数据检查

只对数值指标计算均值、标准差和分位数。`count` 表示非缺失记录的数量；年龄的有效记录数应为 7。分类变量用频数描述。


In [3]:
numeric_columns = ["age_years", "systolic_bp_mmHg", "glucose_mmol_L"]
summary = df[numeric_columns].describe().T
summary["missing_count"] = df[numeric_columns].isna().sum()
display(summary)
display(df["group"].value_counts().rename("sample_count").to_frame())

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)
summary.to_csv(output_dir / "descriptive_statistics.csv", encoding="utf-8-sig")
print("描述性统计已保存到 results/descriptive_statistics.csv")

,count,mean,std,min,25%,50%,75%,max,missing_count
age_years,7.0,40.0000,13.366625,24.0,30.00,38.0,48.500,61.0,1
systolic_bp_mmHg,8.0,127.5000,14.222718,110.0,116.50,126.5,136.750,150.0,0
glucose_mmol_L,8.0,5.3625,0.658868,4.5,4.85,5.3,5.875,6.3,0


,sample_count
group,
A,4
B,4


描述性统计已保存到 results/descriptive_statistics.csv


In [4]:
assert len(df) == 8, "样本数与教学数据不一致"
assert df["sample_id"].is_unique, "样本编号有重复"
assert df["age_years"].isna().sum() == 1, "年龄缺失记录数与教学数据不一致"
assert summary.loc["age_years", "count"] == 7
assert set(df["group"].dropna().unique()) == {"A", "B"}
print("检查通过：读取代码和统计代码均已真实运行。")

检查通过：读取代码和统计代码均已真实运行。


## AI 解释代码与问题识别

以下是一个用于讨论的错误处理示例，请不要把它作为正式分析方法：

```python
age_mean = df["age_years"].fillna(0).mean()
```

这段代码先把缺失年龄替换为 0，再计算均值。但缺失年龄不等于 0 岁，这会改变均值并造成偏差。
本练习用 `mean()` 计算已有年龄记录的均值，同时报告缺失数量；这也不代表缺失问题已经解决。正式研究需要分析缺失原因，再决定处理方法。

请你自己补充：哪些列不能直接求均值？如果读取失败，你会先检查什么？


In [5]:
observed_mean = df["age_years"].mean()
zero_filled_mean = df["age_years"].fillna(0).mean()
print(f"已有年龄记录的均值：{observed_mean:.2f} 岁")
print(f"误把缺失值填为 0 的均值：{zero_filled_mean:.2f} 岁")

已有年龄记录的均值：40.00 岁
误把缺失值填为 0 的均值：35.00 岁
